# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AyeshaShahid-creator/Internship-flyrank-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a scoring task, not classification, clustering, or ranking in the strict sense. The goal isn't to sort pages into fixed categories (classification) or to group similar pages together (clustering). Instead, each page gets a continuous "opportunity score" that reflects how worth refreshing it is, based on signals like traffic trend, search position, and content age. Once every page has a score, sorting by that score naturally produces a ranking — but the model's actual job is to produce the score, not the order itself.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

A fixed rule like "flag if traffic dropped more than X%" only looks at one signal at a time and treats every page the same way. But a page with a huge traffic drop and zero search demand isn't actually an opportunity — it's dead weight. And a page with a moderate drop, strong demand, and a position close to page 1 might be a bigger win than a page with a bigger drop but no realistic path back. ML can learn the right weighted combination of demand, trend, and position from patterns in the data, instead of relying on one hard-coded threshold that treats every page's story the same. That's exactly why my top-10 and bottom-10 outputs looked so different — the score is balancing three signals together, not just reading one number.

In [18]:
# Normalize each signal to 0-1 so they're comparable
demand_score = df["search_volume"].rank(pct=True)
decline_score = (-df["trend_pct"]).rank(pct=True)  # bigger drop = higher score
position_score = df["avg_position"].apply(
    lambda p: 1 - abs(p - 12) / 50 if pd.notna(p) else 0
)  # pages sitting around position ~12 (page 2) score highest — realistic to push to page 1

df["opportunity_score"] = (
    demand_score * 0.4 +
    decline_score * 0.35 +
    position_score * 0.25
)

demand_score = df["search_volume"].rank(pct=True)
decline_score = (-df["trend_pct"]).rank(pct=True)
position_score = df["avg_position"].apply(
    lambda p: 1 - abs(p - 12) / 50 if pd.notna(p) else 0
)

df["opportunity_score"] = (
    demand_score * 0.4 +
    decline_score * 0.35 +
    position_score * 0.25
)
df[["content_id", "search_volume", "trend_pct", "avg_position", "opportunity_score"]] \
    .sort_values("opportunity_score", ascending=False) \
    .head(10)


,content_id,search_volume,trend_pct,avg_position,opportunity_score
1659,content_bbca724138f2,1600.0,-100.0,12.1,0.983846
12994,content_43d0fe93a4a3,1900.0,-100.0,12.4,0.983312
25158,content_51bcdbbb0b67,2900.0,-100.0,10.8,0.980990
20773,content_611615ca6d4a,4400.0,-100.0,9.6,0.976385
10260,content_28ce9207afa4,1900.0,-100.0,10.0,0.975312
12008,content_b1bc831deff1,4400.0,-93.4,12.1,0.972964
19131,content_a0e0cffaef37,4400.0,-100.0,15.2,0.972385
28573,content_3f7dbbd55f0c,14800.0,-98.3,10.2,0.971015
7032,content_a060b47e2355,480.0,-100.0,10.9,0.969054
15248,content_a3aee9f9c8ac,5400.0,-100.0,8.0,0.968828


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The metric I'll use is Precision@50: out of the top 50 pages my model flags as high-priority, what fraction are genuinely worth refreshing? I'm choosing this over something like overall accuracy because the content team only has time to act on a limited number of pages at once — what matters is whether the top of the list is trustworthy, not whether the model is right about every single page in the dataset. A good result means most of those top 50 picks are real opportunities, not false alarms that waste the team's time.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row in this dataframe = one page (URL) on the site, along with its search performance metrics (impressions, clicks, position, CTR, etc.) and content attributes for that time window. This is the unit of analysis because the decision I'm supporting — "should this page be refreshed?" — is made at the page level, not the query level or the site level.

In [20]:
!git clone https://github.com/AyeshaShahid-creator/Internship-flyrank-ML.git
%cd Internship-flyrank-ML


Cloning into 'Internship-flyrank-ML'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 125 (delta 39), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 1.83 MiB | 15.58 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/Internship-flyrank-ML/Internship-flyrank-ML/Internship-flyrank-ML


In [21]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [22]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [23]:
demand_score = df["search_volume"].rank(pct=True)
decline_score = (-df["trend_pct"]).rank(pct=True)  # bigger drop = higher score
position_score = df["avg_position"].apply(
    lambda p: 1 - abs(p - 12) / 50 if pd.notna(p) else 0
)  # pages sitting around position ~12 (page 2) score highest — realistic to push to page 1

df["opportunity_score"] = (
    demand_score * 0.4 +
    decline_score * 0.35 +
    position_score * 0.25
)

demand_score = df["search_volume"].rank(pct=True)
decline_score = (-df["trend_pct"]).rank(pct=True)
position_score = df["avg_position"].apply(
    lambda p: 1 - abs(p - 12) / 50 if pd.notna(p) else 0
)

df["opportunity_score"] = (
    demand_score * 0.4 +
    decline_score * 0.35 +
    position_score * 0.25
)
print("BOTTOM 10 (should look like low-priority pages):")
df[["content_id", "search_volume", "trend_pct", "avg_position", "opportunity_score"]] \
    .sort_values("opportunity_score", ascending=True) \
    .head(10)

BOTTOM 10 (should look like low-priority pages):


,content_id,search_volume,trend_pct,avg_position,opportunity_score
2970,content_3e087a5d8f15,10.0,0.0,138.8,-0.082090
15405,content_a023517539fe,0.0,27907.3,85.8,-0.038471
27916,content_66b4d6070931,0.0,290.3,86.0,-0.034408
11825,content_b69c30792e55,0.0,224.0,84.0,-0.022764
12336,content_b0d74ee35a20,0.0,114.1,85.5,-0.021379
16138,content_d58dd4ba9247,0.0,516.7,82.4,-0.019202
13970,content_ee6b25664a03,0.0,143.8,83.9,-0.016773
28469,content_8a234f7d62b2,0.0,434.0,81.9,-0.016091
9364,content_bcf8e8e2280d,0.0,145.4,83.1,-0.012898
26181,content_4d9ab09aad97,0.0,393.4,80.0,-0.005986


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "flag if traffic dropped more than X%" only looks at one signal at a time and treats every page the same way. But a page with a huge traffic drop and zero search demand isn't actually an opportunity — it's dead weight. And a page with a moderate drop, strong demand, and a position close to page 1 might be a bigger win than a page with a bigger drop but no realistic path back. ML can learn the right weighted combination of demand, trend, and position from patterns in the data, instead of relying on one hard-coded threshold that treats every page's story the same. That's exactly why my top-10 and bottom-10 outputs looked so different — the score is balancing three signals together, not just reading one number

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.